In [1]:
import confnotebook

In [2]:
from pathlib import Path

source = Path("../examples/test/bag_15062026/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 000-591590 2024 Р±РЅ
[1] 000-591590 2025 Р±РЅ
[2] 000-591590 2026 Р±РЅ
[3] 04062001scan
[4] 1 квартал 2026
[5] 113130_Черно-бел. док-т_04062026
[6] 20260604scan150446
[7] 3 кв. 2025
[8] Scan_0034
[9] scan_20010116015111
[10] АС АО КУРГАНМАШЗАВОД
[11] АС РБ за 2025 г
[12] АС СУЭК апрель 26
[13] АСВ Лоста на 31.03.
[14] Акт Сверки РУСАЛ Ачинск
[15] Акт сверки 01012026-31032026
[16] Акт сверки 2025 от 16.04.2026 (1)
[17] Акт сверки 2026 от 16.04.2026
[18] Акт сверки взаиморасчетов № 05800004074 от 29 мая 2026 г
[19] Акт сверки взаиморасчетов № 102 от 03 июня 2026 г
[20] Акт сверки взаиморасчетов № 151 от 20 мая 2026 г без 9_10 этапа
[21] Акт сверки взаиморасчетов № 6171 от 02 июня 2026 г
[22] Акт сверки май 2026
[23] Акт сверки янв-мар 2026 (подпись К-ПМ)
[24] Акт сверки №1523 от 21.04.26 - 5f1bfca4-fde6-48ac-93cd-9266c8fa14d2
[25] Акт сверки №И--000238 от 06.04.26 РИТС
[26] Акт сверки
[27] Альфа-Металл
[28] БАЙКАЛ АКВА Акт сверки 2025-2026.05.28 от КП Крокус (002)
[29] Байкал Аква 23

In [3]:
IDX_FILE = 26

In [4]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
# output_dir = f"../examples/output/{file.stem}"

# debug_image_observer = DebugImageObserver(output_dir=output_dir)

/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [5]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline()

document = pipeline.build(file.read_bytes())

/mnt/data/projects/rusal_recon_srv/repo/recon_vision/.venv/lib/python3.11/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv5_server_det', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/PP-OCRv5_server_det')
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', '/mnt/data/projects/rusal_recon_srv/repo/recon_vision/models/cyrillic_PP-OCRv5_mobile_rec')
2026-06-17 17:42:56.506 | INFO     | vision_core.pipelines.build_document:build:104 - Обработка страницы 0 с dpi 200...
2026-06-17 17:42:56.566 | INFO     | vision_core.pipelines.build_document:_process_page:186 - Коррекция ориентации и наклона...
2026-06-17 17:42:56.694 | DEBUG    | vision_core.preprocessor.image_orientation:process:47 - Ориентация страницы: 0° 

In [6]:
def get_cell_covering(table, row: int, col: int):
    for cell in table.get_rows()[row]:
        if cell.col <= col < cell.col + cell.colspan:
            return cell
    return None


summary_text = ""
summary_cell_text: list[str] = []
for page in document.pages:
    text_paragraph = " ".join(paragraph.text for paragraph in page.paragraphs)
    for table in page.tables:
        if table.continuation_of is not None:
            continue
        dc = table.dc_cols
        num_row = table.get_dc_header_row()
        if num_row == -1 or not dc:
            continue
        seen_cells: set[int] = set()
        for i, col in enumerate(sorted(dc)):
            for j in range(num_row):
                cell = get_cell_covering(table, j, col)
                if cell is None or id(cell) in seen_cells:
                    continue
                seen_cells.add(id(cell))
                cell_text = cell.value.strip()
                if cell_text:
                    summary_cell_text.append(cell_text)

    summary_text += text_paragraph + " "

print("SUMMARY CELL TEXT:")
print(summary_cell_text)
print("\nSUMMARY TEXT:")
print(summary_text)

2026-06-17 17:43:02.283 | DEBUG    | vision_core.postprocessor.dc_cols_resolver:_is_similar_keyword:52 - Проверяем похожесть 'дебет' и 'кредит' (ratio=0.33)


SUMMARY CELL TEXT:
['По данным федерального государственного образовательного бюджетного учреждения\nвысшего образования "Финансовый университет при Правительстве Российской Федерации".\npy6.']

SUMMARY TEXT:
взаимных расчетов №º 0000-0000389 OT 15.04.2026 Акт сверки за периодЯнварь 2025 г. - Январь 2026 г. можду федеральным государственным образовательным бюджетным учреждением высшего образования "Финансовый университет при Правительстве Российской Федерации" и ПАО "РУСАЛ БРАТСК" по договору РБ-Д-25-776 от 08.09.2025 Мы, нижеподписавшиеся, Заместитель проректора по экономической и финансовой работе Гапонова Н. А. от федерального государственного образовательного бюджетного учреждения высшего образования "Финансовый университет при Правительстве Российской Федерации", с одной стороны, и ОТ ПАО "РУСАЛ Братск", с другой стороны, составили настоящий акт сверки в том, что состояние взаимных расчетов по данным учета следующее: По данным федерального государственного бюджетного учреждения вы

In [7]:
import re

_LAT2CYR = str.maketrans({
    'A':'А','B':'В','C':'С','E':'Е','H':'Н','K':'К','M':'М','O':'О','P':'Р','T':'Т','X':'Х','Y':'У',
    'a':'а','c':'с','e':'е','o':'о','p':'р','x':'х','y':'у','k':'к','m':'м','h':'н','b':'в','t':'т',
    'R':'Р','r':'р','V':'В','v':'в'
})
_QUOTES = '«»\u201c\u201d\u201e\u2018\u2019\u201a\u2039\u203a'
_QUOTE_NORM = str.maketrans(_QUOTES, '"' * len(_QUOTES))

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.translate(_QUOTE_NORM)
    text = re.sub(r'"{2,}', '"', text)          # "" -> "
    text = re.sub(r'(\S)"', r'\1 "', text)      # ОБЩЕСТВО" -> ОБЩЕСТВО "
    text = text.translate(_LAT2CYR)
    text = text.upper().replace('Ё', 'Е')
    text = re.sub(r'\b000\b', 'ООО', text)
    text = re.sub(r'\s+', ' ', text, flags=re.UNICODE)
    return text.strip()



normalized_text = normalize_text(summary_text)

summary_cell_text_norm = [normalize_text(cell) for cell in summary_cell_text]


print("SUMMARY CELL TEXT:")
print(summary_cell_text_norm)
print("\nSUMMARY TEXT:")
print(normalized_text)

SUMMARY CELL TEXT:
['ПО ДАННЫМ ФЕДЕРАЛЬНОГО ГОСУДАРСТВЕННОГО ОБРАЗОВАТЕЛЬНОГО БЮДЖЕТНОГО УЧРЕЖДЕНИЯ ВЫСШЕГО ОБРАЗОВАНИЯ "ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ ". РУ6.']

SUMMARY TEXT:
ВЗАИМНЫХ РАСЧЕТОВ №º 0000-0000389 ОТ 15.04.2026 АКТ СВЕРКИ ЗА ПЕРИОДЯНВАРЬ 2025 Г. - ЯНВАРЬ 2026 Г. МОЖДУ ФЕДЕРАЛЬНЫМ ГОСУДАРСТВЕННЫМ ОБРАЗОВАТЕЛЬНЫМ БЮДЖЕТНЫМ УЧРЕЖДЕНИЕМ ВЫСШЕГО ОБРАЗОВАНИЯ "ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ " И ПАО "РУСАЛ БРАТСК " ПО ДОГОВОРУ РБ-Д-25-776 ОТ 08.09.2025 МЫ, НИЖЕПОДПИСАВШИЕСЯ, ЗАМЕСТИТЕЛЬ ПРОРЕКТОРА ПО ЭКОНОМИЧЕСКОЙ И ФИНАНСОВОЙ РАБОТЕ ГАПОНОВА Н. А. ОТ ФЕДЕРАЛЬНОГО ГОСУДАРСТВЕННОГО ОБРАЗОВАТЕЛЬНОГО БЮДЖЕТНОГО УЧРЕЖДЕНИЯ ВЫСШЕГО ОБРАЗОВАНИЯ "ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ ", С ОДНОЙ СТОРОНЫ, И ОТ ПАО "РУСАЛ БРАТСК ", С ДРУГОЙ СТОРОНЫ, СОСТАВИЛИ НАСТОЯЩИЙ АКТ СВЕРКИ В ТОМ, ЧТО СОСТОЯНИЕ ВЗАИМНЫХ РАСЧЕТОВ ПО ДАННЫМ УЧЕТА СЛЕДУЮЩЕЕ: ПО ДАННЫМ ФЕДЕРАЛЬНОГО ГОСУДАРСТВЕННОГО БЮДЖЕТНОГО УЧРЕЖДЕНИЯ

In [10]:
ORGFORMS = {
    'АО':   'акционерное общество',
    'ОАО':  'открытое акционерное общество',
    'ЗАО':  'закрытое акционерное общество',
    'ООО':  'общество с ограниченной ответственностью',
    'ИП':   'индивидуальный предприниматель',
    'ПАО':  'публичное акционерное общество',
    'НП':   'некоммерческое партнерство',
    'ГУП':  'государственное унитарное предприятие',
    'МУП':  'муниципальное унитарное предприятие',
    'ФГУП': 'федеральное государственное унитарное предприятие',
    "ФГОБУ ВО": "ФЕДЕРАЛЬНОЕ ГОСУДАРСТВЕННОЕ ОБРАЗОВАТЕЛЬНОЕ БЮДЖЕТНОЕ УЧРЕЖДЕНИЕ ВЫСШЕГО ОБРАЗОВАНИЯ",
}
ORGFORMS_FULL2SHORT = {v.upper(): k for k, v in ORGFORMS.items()}
def ocr_robust(s: str) -> str:
    """
    Заменяет букву "О" на паттерн, который может соответствовать как "О",
    так и "0", для повышения устойчивости к ошибкам OCR.
    """
    return s.replace('О', '[О0]').replace('о', '[о0]')

org_forms_pattern = "|".join(
    ocr_robust(k) for k in sorted(ORGFORMS.keys(), key=len, reverse=True)
) + r'|(?:' + "|".join(
    ocr_robust(v) for v in ORGFORMS.values()
) + r')'

In [13]:
def _levenshtein_distance(a: str, b: str) -> int:
    if len(a) < len(b):
        a, b = b, a
    previous = list(range(len(b) + 1))
    for i, ch_a in enumerate(a, start=1):
        current = [i]
        for j, ch_b in enumerate(b, start=1):
            cost = 0 if ch_a == ch_b else 1
            current.append(
                min(
                    previous[j] + 1,
                    current[j - 1] + 1,
                    previous[j - 1] + cost,
                )
            )
        previous = current
    return previous[-1]

def _similarity_ratio(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    dist = _levenshtein_distance(a, b)
    return 1.0 - dist / max(len(a), len(b))

def fix_rusal(name: str) -> str:
    return re.sub(r'РУСАЛ([А-ЯЁ])', r'РУСАЛ \1', name)

def find_org_form_fuzzy(text: str, threshold: float = 0.70) -> list[tuple[str, str, int]]:
    """Быстро находит все орг.формы в тексте."""
    results = []
    text_upper = text.upper()
    
    candidates = []
    
    # Краткие формы (2-4 буквы)
    for match in re.finditer(r'\b[А-ЯЁ]{2,4}\b', text_upper):
        candidates.append((match.group(), match.start()))
    
    # Длинные формы (несколько слов заглавными буквами)
    for match in re.finditer(r'\b(?:[А-ЯЁ][А-ЯЁа-яё]*\s+){2,8}[А-ЯЁ][А-ЯЁа-яё]*\b', text_upper):
        candidates.append((match.group(), match.start()))
    
    for candidate, pos in candidates:
        best_match = None
        best_ratio = 0.0
        
        # Проверяем краткие формы
        for short_form in ORGFORMS.keys():
            if len(candidate) == len(short_form):
                ratio = _similarity_ratio(candidate, short_form)
                if ratio > best_ratio and ratio >= threshold:
                    best_ratio = ratio
                    best_match = short_form
        
        # Проверяем полные формы
        for short_form, full_form in ORGFORMS.items():
            full_upper = full_form.upper()
            if abs(len(candidate) - len(full_upper)) < len(full_upper) * 0.3:
                ratio = _similarity_ratio(candidate, full_upper)
                if ratio > best_ratio and ratio >= threshold:
                    best_ratio = ratio
                    best_match = short_form
        
        if best_match:
            results.append((best_match, candidate, pos))
    
    return results

def extract_org_names(text: str, threshold: float = 0.70) -> list[str]:
    """Извлекает уникальные полные названия организаций."""
    results: list[str] = []
    seen_full: set[str] = set()
    seen_names: set[str] = set()
    
    # Находим все орг.формы в тексте
    org_forms = find_org_form_fuzzy(text, threshold)
    
    for short_form, matched_text, form_pos in org_forms:
        form_end = form_pos + len(matched_text)
        
        # Ищем БЛИЖАЙШЕЕ название в кавычках ДО формы
        before = text[max(0, form_pos - 200):form_pos]
        quotes_before = list(re.finditer(r'"([^"]+)"', before))
        
        # Ищем БЛИЖАЙШЕЕ название в кавычках ПОСЛЕ формы
        after = text[form_end:form_end + 200]
        quotes_after = list(re.finditer(r'"([^"]+)"', after))
        
        # Берем только ближайшее название
        nearest_name = None
        
        if quotes_before and quotes_after:
            # Сравниваем расстояния
            dist_before = form_pos - quotes_before[-1].end()
            dist_after = quotes_after[0].start()
            
            if dist_before <= dist_after:
                nearest_name = quotes_before[-1].group(1)
            else:
                nearest_name = quotes_after[0].group(1)
        elif quotes_before:
            nearest_name = quotes_before[-1].group(1)
        elif quotes_after:
            nearest_name = quotes_after[0].group(1)
        
        if nearest_name:
            name = fix_rusal(nearest_name.strip()).replace('"', '')
            if name:
                full_name = name + ', ' + short_form
                
                if full_name not in seen_full:
                    seen_full.add(full_name)
                    seen_names.add(name)
                    results.append(full_name)
    
    # Проход 2: Поиск "РУСАЛ..." без орг.формы
    for q in re.finditer(r'"(РУСАЛ[^"]*)"', text):
        name = fix_rusal(q.group(1).strip()).replace('"', '')
        if name not in seen_names:
            seen_names.add(name)
            results.append(name + ", ")
    
    return results



print("Из ячеек:", extract_org_names(" ".join(summary_cell_text_norm)))
print("\nИз полного текста:", extract_org_names(normalized_text))


Из ячеек: ['ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ, ФГОБУ ВО']

Из полного текста: ['РУСАЛ БРАТСК, ПАО', 'ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ, ФГОБУ ВО']


In [14]:
from enum import Enum


class Role(Enum):
    BUYER = 0
    SELLER = 1
    UNKNOWN = -1


ROLE_GLOSSARY = {
    r'\bОТ ПОКУПАТЕЛЯ\b': 'BUYER_SOURCE',
    r'\bОТ ПРОДАВЦА\b':   'SELLER_SOURCE',
    r'\bМЕЖДУ\b':         'PARTICIPATION_SCOPE',
}


def _org_token(full_name: str) -> str:
    return full_name.split(",", 1)[0].strip()


def _find_events(text: str) -> list[dict]:
    found = []
    for pattern, role in ROLE_GLOSSARY.items():
        for m in re.finditer(pattern, text, re.IGNORECASE):
            found.append({"keyword": m.group(), "role": role,
                          "start_index": m.start(), "end_index": m.end()})
    return sorted(found, key=lambda x: x["start_index"])


def _find_orgs_in_span(text: str, left: int, right: int, orgs: list[str]) -> list[str]:
    span = text[left:right]
    return [o for o in orgs if _org_token(o) and _org_token(o) in span]


def _find_working_pair(text: str, orgs: list[str], events: list[dict]) -> list[str]:
    """Два контрагента рядом с якорем МЕЖДУ."""
    anchor = next((e for e in events if e["role"] == "PARTICIPATION_SCOPE"), None)
    if anchor:
        window = text[anchor["end_index"]: anchor["end_index"] + 400]
        hits = sorted(
            [(window.find(_org_token(o)), o) for o in orgs if _org_token(o) in window]
        )
        pair = [o for _, o in hits[:2]]
        if len(pair) == 2:
            return pair
    return orgs[:2]


def _apply_symmetry(roles: dict) -> None:
    buyers   = [o for o, r in roles.items() if r == Role.BUYER]
    sellers  = [o for o, r in roles.items() if r == Role.SELLER]
    unknowns = [o for o, r in roles.items() if r == Role.UNKNOWN]
    if buyers and unknowns and not sellers:
        for o in unknowns:
            print(f"Правило симметрии: {o} -> SELLER")
            roles[o] = Role.SELLER
    elif sellers and unknowns and not buyers:
        for o in unknowns:
            print(f"Правило симметрии: {o} -> BUYER")
            roles[o] = Role.BUYER

def _deduplicate_orgs(orgs: list[str]) -> list[str]:
    tokens = [_org_token(o) for o in orgs]
    return [
        org for i, org in enumerate(orgs)
        if not any(tokens[i] in tokens[j] and tokens[i] != tokens[j] for j in range(len(tokens)))
    ]

def assign_roles(text: str, orgs: list[str]) -> dict[str, Role]:
    events = _find_events(text)
    print("Найдены события:", events)
    pair   = _find_working_pair(text, orgs, events)
    print("Рабочая пара:", pair)
    roles  = {o: Role.UNKNOWN for o in pair}

    # 1) РУСАЛ -> всегда BUYER (жёсткое правило)
    for o in pair:
        if "РУСАЛ" in _org_token(o):
            roles[o] = Role.BUYER
            print(f"Правило: {o} содержит 'РУСАЛ' -> BUYER")
    _apply_symmetry(roles)

    # 2) Явные якоря ОТ ПОКУПАТЕЛЯ / ОТ ПРОДАВЦА (только для UNKNOWN)
    for idx, event in enumerate(events):
        if event["role"] not in {"BUYER_SOURCE", "SELLER_SOURCE"}:
            continue
        next_start = events[idx + 1]["start_index"] if idx + 1 < len(events) else len(text)
        target = Role.BUYER if event["role"] == "BUYER_SOURCE" else Role.SELLER
        for o in _find_orgs_in_span(text, event["end_index"], next_start, pair):
            if roles[o] == Role.UNKNOWN:
                roles[o] = target
                print(f"Правило: {o} находится в диапазоне {event['keyword']} -> {target.name}")
    _apply_symmetry(roles)

    # 3) Позиционный фоллбек: первый -> SELLER, второй -> BUYER
    unknowns = [o for o, r in roles.items() if r == Role.UNKNOWN]
    if unknowns:
        roles[unknowns[0]] = Role.SELLER
        print(f"Правило позиционного фоллбека: {unknowns[0]} -> SELLER")
        for o in unknowns[1:]:
            roles[o] = Role.BUYER
            print(f"Правило позиционного фоллбека: {o} -> BUYER")

    return roles


# --- запуск ---
orgs_name = extract_org_names(" ".join(summary_cell_text_norm))
orgs_name_full = extract_org_names(normalized_text)

if len(orgs_name) >= 2:
    orgs_name_full_filtered = orgs_name
elif orgs_name:
    orgs_name_set = set(orgs_name)
    orgs_name_full_filtered = [
        o for o in orgs_name_full if any(n in o for n in orgs_name_set)
    ]
else:
    orgs_name_full_filtered = orgs_name_full

orgs_name_full_filtered = _deduplicate_orgs(orgs_name_full_filtered)
print("Организации (после дедупликации):", orgs_name_full_filtered)


roles_by_org = assign_roles(normalized_text, orgs_name_full_filtered)

for org, role in roles_by_org.items():
    print(f"- {org} -> {role.name}")

buyers  = [o for o, r in roles_by_org.items() if r == Role.BUYER]
sellers = [o for o, r in roles_by_org.items() if r == Role.SELLER]
print("\nBUYER:", buyers)
print("SELLER:", sellers)


Организации (после дедупликации): ['ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ, ФГОБУ ВО']
Найдены события: []
Рабочая пара: ['ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ, ФГОБУ ВО']
Правило позиционного фоллбека: ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ, ФГОБУ ВО -> SELLER
- ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ, ФГОБУ ВО -> SELLER

BUYER: []
SELLER: ['ФИНАНСОВЫЙ УНИВЕРСИТЕТ ПРИ ПРАВИТЕЛЬСТВЕ РОССИЙСКОЙ ФЕДЕРАЦИИ, ФГОБУ ВО']
